# Lesson 5 : Foundry Tools

In Lesson 4, we have learned basic tool calling by using built-in hosted tools in Agent Framework.  
As I have mentioned in Lesson 1, ```AzureAIClient``` is based on ```azure-ai-projects``` v2 SDK. You, therefore, can also integrate with a wide variety of Microsoft Foundry tools - such as, Azure AI Search, Microsoft SharePoint, Microsoft Fabric, and 3rd party tools -, when you're using ```AzureAIClient```.<br>
Collaborating with tools in Foundry, you can take a variety of benefits (a lot of perconfigured tools, identity integration, etc) from Microsoft ecosystem.

In this exercise, we change 2 examples in Lesson 4 to use "Grounding with Bing Search" tool and "MCP" tool in Microsoft Foundry, respectively.

> Note : Some tools - such as, SharePoint tool, Fabric tool, 3rd-party MCP tools, etc - handle the identity.  
> When you need to provide the credential or identity (such as, OAuth access token) in tools, process authentication and authorization in your apps (such as, showing the login UI, processing OAuth flow, etc) and get the identity for tool's usage.

## Web Search tool in Foundry Tools

In the first example, we change the example by ```HostedWebSearchTool``` in Lesson 4 to the example to use "Grounding with Bing Search" in Foundry tools.

### 1. Create and connect to Bing Grounding Search resource

Before writing code, please create and connect to "Grounding with Bing Search" resource  in Microsoft Foundry as follows.

1. Open [Azure Portal](https://portal.azure.com) and create a new "Grounding with Bing Search" resource.
2. Open Foundry Portal (new portal), click "Operate" menu, and select "Admin" in left-side navigation.
3. Select your project, and connect to above "Grounding with Bing Search" resource by clicking "Add connection".

You can then see the connected resource name for "Grounding with Bing Search". (This connected resource name is used later.)

### 2. Run with "Grounding with Bing Search" tool

First we create a ```AzureAIClient``` object as usual.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we create an agent with "Grounding with Bing Search" tool's configuration as follows.

This configuration is the same as the configuration used by [REST API](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/bing-tools?view=foundry&tabs=grounding-with-bing&pivots=rest) which is used in ```azure-ai-projects```.

You already have the connected resource name for Bing Grounding search (see above) and **please replace the following connected resource id with your settings**.<br>
This id is obtained as follows.

- Select "Build" tab in Foundry Portal.
- Select "Tools" menu.
- Click the tool you use.

In [2]:
from agent_framework import Agent

# ToDo : fill your below settings
#       (e.g., /subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME})
CONNECTED_RESOURCE_ID = "xxxxxxxxxxxxxxxxxxxxxx"

agent = Agent(
    name="WeatherAgentWithBingSearch",
    client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[
        {
            "type": "bing_grounding",
            "bing_grounding": {
                "search_configurations": [
                    {
                        "project_connection_id": CONNECTED_RESOURCE_ID
                    }
                ]
            }
        }
    ],
)

Now we run the agent as follows. (Use [tracing](./02_trace.ipynb) and see that tool is used internally.)

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")  # "今日の大阪の天気と気温を教えてください。"
display(Markdown(result.text))

Osaka today is **mostly clear**. The **current temperature is about 6°C**, with a **low around 1°C** expected (so it stays chilly).【5:3†source】

## MCP tool in Foundry Tools

In the next example, we change the example by ```HostedMCPTool``` in Lesson 4 to the example to use MCP tool in Foundry tools.

### 1. Connect to Microsoft Learn in Foundry tools

Before writing code, please connect to "Microsoft Learn" tool in Microsoft Foundry as follows.

1. Open Foundry Portal.
2. Go to "Build" tab.
3. Select "Tools" menu.
4. Click "Connect a tool"
5. Select "Microsoft Learn" in catalog, and establish connection.

### 2. Run with "Microsoft Learn" tool in Foundry

Now we create an agent with above.

"Microsoft Learn" tool is one of MCP tools in Foundry.<br>
Same as above, please configure properties for MCP tool, which is the same configuration in ```azure-ai-projects```.

**Please replace the following connected resource id with your settings**.<br>
Same as above, this id is obtained as follows.

- Select "Build" tab in Foundry Portal.
- Select "Tools" menu.
- Click the tool you use.

In [4]:
from agent_framework import Agent

# ToDo : fill your below settings
#       (e.g., /subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME})
CONNECTED_RESOURCE_ID = "xxxxxxxxxxxxxxxxxxxxxx"

agent = Agent(
    name="MSTechKnowledgeAgentWithFoundry",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",  # "あなたは、Microsoft の製品やサービスに関する技術質問に答える Agent です。"
    tools=[
        {
            "type": "mcp",
            "server_label": "mymcp01",
            "server_url": "https://learn.microsoft.com/api/mcp",
            "require_approval": "never",
            "project_connection_id": CONNECTED_RESOURCE_ID,
        }
    ],
)

Let's ask a technical question about Microsoft Azure.

In [5]:
from IPython.display import Markdown, display

result = await agent.run("How to create an Azure storage account using Azure CLI ?")  # "Azure CLI を使って Azure Storage アカウントを作成する方法を教えてください。"
display(Markdown(result.text))

To create an Azure Storage account with **Azure CLI**, do:

```bash
# Sign in (skip if using Cloud Shell)
az login

# Create a resource group
az group create \
  --name storage-resource-group \
  --location eastus

# Create a StorageV2 (general-purpose v2) storage account
# NOTE: --name must be globally unique (3-24 chars, lowercase letters/numbers)
az storage account create \
  --name <account-name> \
  --resource-group storage-resource-group \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Verify:

```bash
az storage account show \
  --name <account-name> \
  --resource-group storage-resource-group
```

Reference: https://learn.microsoft.com/en-us/azure/storage/common/storage-account-create#create-a-storage-account

## Final note

All tools registered in your Foundry project has unique connected resource id, and your agents built in Agent Framework can then connect to any tools in project by setting this id.

For available tool's type and configurations, see [API reference](https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models?view=azure-python-preview) in Azure AI Projects SDK.<br>
(As I have mentioned in Lesson 1, ```AzureAIClient``` uses Azure AI Projects SDK (```azure-ai-projects```) at the bottom.)

> Note : Mostly 3rd party tools in Microsoft Foundry has MCP type.